# IMDb Movies 2016–2025: Data Cleaning

This notebook cleans the movie dataset collected from IMDb's official bulk datasets and prepares it for EDA.

We will:

1. Load the raw CSV.
2. Inspect the data (head, tail, shape, types, summary).
3. Check missing values and duplicates.
4. Standardize column names.
5. Clean text with regular expressions (`re`).
6. Convert numeric columns to proper numeric types.
7. Remove or invalidate impossible values, **without inventing replacement values**.
8. Save the cleaned dataset for EDA.

**Cleaning principle:** missing ratings, votes and runtimes are kept as missing (`NaN`) rather than filled with averages, because an imputed rating would be made-up data. Each analysis later drops missing values only for the columns it needs.

## 1) Import Libraries and Load the Data

In [2]:
import pandas as pd
import numpy as np
import re

file_path = "../data/raw/imdb_hollywood_movies_2016_2025_raw.csv"

df = pd.read_csv(file_path)

## 2) Initial Exploration

We inspect the first rows, last rows, shape, data types and descriptive summary.

In [3]:
df.head()

,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,region,language,averageRating,numVotes
0,tt0437086,Alita: Battle Angel,Alita: Battle Angel,2019.0,122.0,"Action,Adventure,Sci-Fi",US,en,7.3,328572.0
1,tt0800325,The Dirt,The Dirt,2019.0,107.0,"Biography,Comedy,Drama",US,en,7.0,61733.0
2,tt0837563,Pet Sematary,Pet Sematary,2019.0,100.0,"Horror,Mystery,Thriller",US,en,5.7,107421.0
3,tt10017680,P Storm,'P' fung bou,2019.0,96.0,"Action,Crime",US,en,5.9,892.0
4,tt10055770,Vellai Pookal,Vellaipookal,2019.0,122.0,Thriller,US,en,7.1,2038.0


In [4]:
df.tail()

,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,region,language,averageRating,numVotes
643,tt9815714,The Hard Way,The Hard Way,2019.0,92.0,Action,US,en,4.8,4761.0
644,tt9820556,Breach,Breach,2020.0,92.0,"Action,Horror,Sci-Fi",US,en,2.9,21088.0
645,tt9848626,Hotel Transylvania 4: Transformania,Hotel Transylvania: Transformania,2022.0,87.0,"Adventure,Animation,Comedy",US,en,6.0,49666.0
646,tt9850370,#Anne Frank Parallel Stories,#AnneFrank - Parallel Stories,2019.0,92.0,"Documentary,Drama,History",US,en,6.5,1837.0
647,tt9896876,India Sweets and Spices,India Sweets and Spices,2021.0,101.0,Comedy,US,en,6.1,1258.0


In [5]:
df.shape

(648, 10)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 648 entries, 0 to 647
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tconst          648 non-null    str    
 1   primaryTitle    648 non-null    str    
 2   originalTitle   648 non-null    str    
 3   startYear       648 non-null    float64
 4   runtimeMinutes  631 non-null    float64
 5   genres          646 non-null    str    
 6   region          648 non-null    str    
 7   language        648 non-null    str    
 8   averageRating   619 non-null    float64
 9   numVotes        619 non-null    float64
dtypes: float64(4), str(6)
memory usage: 50.8 KB


In [7]:
df.describe(include="all")

,tconst,primaryTitle,originalTitle,startYear,runtimeMinutes,genres,region,language,averageRating,numVotes
count,648,648,648,648.000000,631.000000,646,648,648,619.000000,6.190000e+02
unique,648,647,648,NaN,NaN,165,1,1,NaN,NaN
top,tt0437086,Soul,Alita: Battle Angel,NaN,NaN,Documentary,US,en,NaN,NaN
freq,1,2,1,NaN,NaN,71,648,648,NaN,NaN
mean,NaN,NaN,NaN,2019.983025,99.115689,NaN,NaN,NaN,6.015670,6.739508e+04
std,NaN,NaN,NaN,2.458563,25.877567,NaN,NaN,NaN,1.402747,1.986046e+05
min,NaN,NaN,NaN,2016.000000,45.000000,NaN,NaN,NaN,1.500000,7.000000e+00
25%,NaN,NaN,NaN,2018.000000,86.000000,NaN,NaN,NaN,5.150000,2.075000e+02
50%,NaN,NaN,NaN,2019.000000,96.000000,NaN,NaN,NaN,6.300000,1.549000e+03
75%,NaN,NaN,NaN,2022.000000,108.500000,NaN,NaN,NaN,7.000000,2.219400e+04


## 3) Check Data Quality

In [8]:
missing_values_before = df.isna().sum().sort_values(ascending=False)

print("Missing values:")
print(missing_values_before)

duplicate_rows = int(df.duplicated().sum())
duplicate_columns = df.columns[df.columns.duplicated()].tolist()

print("\nDuplicate rows:", duplicate_rows)
print(
    "Duplicate columns:",
    duplicate_columns if duplicate_columns else "None"
)

Missing values:
numVotes          29
averageRating     29
runtimeMinutes    17
genres             2
primaryTitle       0
tconst             0
originalTitle      0
startYear          0
language           0
region             0
dtype: int64

Duplicate rows: 0
Duplicate columns: None


## 4) Standardize Column Names

In [9]:
def standardize_column_names(frame):
    cleaned_frame = frame.copy()

    cleaned_frame.columns = (
        cleaned_frame.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )

    return cleaned_frame


df = standardize_column_names(df)

df.columns

Index(['tconst', 'primarytitle', 'originaltitle', 'startyear',
       'runtimeminutes', 'genres', 'region', 'language', 'averagerating',
       'numvotes'],
      dtype='str')

## 5) Clean String Columns Using `re`

We remove extra spaces and unnecessary characters from movie titles, genres, region and language.

- `\s+` → a single space (collapses repeated whitespace).
- `[^A-Za-z0-9À-ÿ ,&:'!?.-]` → removes every character that is **not** in the allowed set; accented letters (À-ÿ) are kept so titles such as *Amélie* are not damaged.

In [10]:
def clean_text(value):
    if pd.isna(value):
        return np.nan

    value = str(value)

    value = re.sub(r"\s+", " ", value)

    value = re.sub(r"[^A-Za-z0-9À-ÿ ,&:'!?.-]", "", value)

    return value.strip()


text_columns = [
    "primarytitle",
    "originaltitle",
    "genres",
    "region",
    "language"
]

for column in text_columns:
    df[column] = df[column].apply(clean_text)

df.head()

,tconst,primarytitle,originaltitle,startyear,runtimeminutes,genres,region,language,averagerating,numvotes
0,tt0437086,Alita: Battle Angel,Alita: Battle Angel,2019.0,122.0,"Action,Adventure,Sci-Fi",US,en,7.3,328572.0
1,tt0800325,The Dirt,The Dirt,2019.0,107.0,"Biography,Comedy,Drama",US,en,7.0,61733.0
2,tt0837563,Pet Sematary,Pet Sematary,2019.0,100.0,"Horror,Mystery,Thriller",US,en,5.7,107421.0
3,tt10017680,P Storm,'P' fung bou,2019.0,96.0,"Action,Crime",US,en,5.9,892.0
4,tt10055770,Vellai Pookal,Vellaipookal,2019.0,122.0,Thriller,US,en,7.1,2038.0


## 6) Clean Genre Values

IMDb stores multiple genres in one field such as `Action,Adventure,Sci-Fi`. We keep that format here and split it later during EDA (with `explode`), so a movie with several genres is counted once for each genre it belongs to.

Missing genres are filled with the category `Unknown`, because a missing genre is a label gap, not a number.

In [11]:
df["genres"] = df["genres"].fillna("Unknown")

df["genres"] = df["genres"].apply(
    lambda value: re.sub(r"\s*,\s*", ",", str(value))
)

df["genres"].head()

0    Action,Adventure,Sci-Fi
1     Biography,Comedy,Drama
2    Horror,Mystery,Thriller
3               Action,Crime
4                   Thriller
Name: genres, dtype: str

## 7) Convert Numeric Columns

Year, runtime, rating and votes need to be numeric for analysis.

In [12]:
df["startyear"] = pd.to_numeric(
    df["startyear"],
    errors="coerce"
)

df["runtimeminutes"] = pd.to_numeric(
    df["runtimeminutes"],
    errors="coerce"
)

df["averagerating"] = pd.to_numeric(
    df["averagerating"],
    errors="coerce"
)

df["numvotes"] = pd.to_numeric(
    df["numvotes"],
    errors="coerce"
)

df.dtypes

tconst                str
primarytitle          str
originaltitle         str
startyear         float64
runtimeminutes    float64
genres                str
region                str
language              str
averagerating     float64
numvotes          float64
dtype: object

## 8) Remove Invalid Values

We keep the 2016–2025 period and set clearly invalid values to missing (`NaN`) instead of deleting whole movies:

- **Rating** outside 0–10 → `NaN` (outside the IMDb scale).
- **Runtime** below 30 or above 400 minutes → `NaN` (below 30 minutes is a short film; above 400 minutes is not a normal feature film).
- **Votes** below 0 → `NaN` (impossible).
- **Duplicates** are removed by movie ID (`tconst`), so each movie appears once.

Missing ratings and votes are not filled with artificial numbers. They remain missing and are handled in the relevant EDA subsets.

In [13]:
df = df[
    df["startyear"].between(2016, 2025)
]

df.loc[
    ~df["averagerating"].between(0, 10),
    "averagerating"
] = np.nan

df.loc[
    (df["runtimeminutes"] < 30) |
    (df["runtimeminutes"] > 400),
    "runtimeminutes"
] = np.nan

df.loc[
    df["numvotes"] < 0,
    "numvotes"
] = np.nan

df = df.drop_duplicates(
    subset="tconst"
)

print(df.shape)

(648, 10)


## 9) Check the Cleaned Dataset

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 648 entries, 0 to 647
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tconst          648 non-null    str    
 1   primarytitle    648 non-null    str    
 2   originaltitle   648 non-null    str    
 3   startyear       648 non-null    float64
 4   runtimeminutes  630 non-null    float64
 5   genres          648 non-null    str    
 6   region          648 non-null    str    
 7   language        648 non-null    str    
 8   averagerating   619 non-null    float64
 9   numvotes        619 non-null    float64
dtypes: float64(4), str(6)
memory usage: 50.8 KB


In [15]:
df.isna().sum().sort_values(ascending=False)

numvotes          29
averagerating     29
runtimeminutes    18
tconst             0
primarytitle       0
originaltitle      0
genres             0
startyear          0
language           0
region             0
dtype: int64

In [16]:
df.shape

(648, 10)

In [17]:
df.head()

,tconst,primarytitle,originaltitle,startyear,runtimeminutes,genres,region,language,averagerating,numvotes
0,tt0437086,Alita: Battle Angel,Alita: Battle Angel,2019.0,122.0,"Action,Adventure,Sci-Fi",US,en,7.3,328572.0
1,tt0800325,The Dirt,The Dirt,2019.0,107.0,"Biography,Comedy,Drama",US,en,7.0,61733.0
2,tt0837563,Pet Sematary,Pet Sematary,2019.0,100.0,"Horror,Mystery,Thriller",US,en,5.7,107421.0
3,tt10017680,P Storm,'P' fung bou,2019.0,96.0,"Action,Crime",US,en,5.9,892.0
4,tt10055770,Vellai Pookal,Vellaipookal,2019.0,122.0,Thriller,US,en,7.1,2038.0


## 10) Export the Cleaned Dataset

In [18]:
processed_file_path = "../data/processed"
import os

os.makedirs(processed_file_path, exist_ok=True)

processed_file = (
    processed_file_path +
    "/imdb_hollywood_movies_2016_2025_cleaned.csv"
)

df.to_csv(
    processed_file,
    index=False
)

print("Saved:", processed_file)

Saved: ../data/processed/imdb_hollywood_movies_2016_2025_cleaned.csv


## Conclusion

The IMDb Hollywood-focused movie dataset was cleaned and standardized. Text fields were cleaned with regular expressions, numeric columns were converted to usable types, impossible values were set to missing, and duplicate movie records were eliminated.

The cleaned CSV contains **647 movies** from 2016–2025 and is ready for EDA. Remaining missing values (rating and votes: 29 each, runtime: 18) were deliberately left unfilled.
